# 4. NeighborLoader Benchmark

This notebook demonstrates how to load the pre-built `HeteroData` graph and set up a `NeighborLoader`.
This component is responsible for sampling subgraphs for mini-batch training, which is critical for large-scale graphs.

In [ ]:
# Configuration
import os
import torch

# EXTERNAL DRIVE CONFIGURATION
ROOT_DIR = "/Volumes/Backup Plus/Zaman/graph"
HETERO_GRAPH_PATH = os.path.join(ROOT_DIR, "processed_data", "hetero_graph.pt")

# Loader Parameters
BATCH_SIZE = 1024
NUM_NEIGHBORS = [10, 10] # Sample 10 neighbors for 2 hops

# M1 Pro Optimization: Device Selection
DEVICE = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")

# M1 Pro Optimization: DataLoader settings
NUM_WORKERS = 4 
PERSISTENT_WORKERS = True
PIN_MEMORY = True if DEVICE == 'mps' else False

In [ ]:
# Imports
from torch_geometric.data import HeteroData
from torch_geometric.loader import NeighborLoader
from torch_geometric.transforms import ToUndirected
from tqdm.notebook import tqdm
import time

In [ ]:
# Load Graph
if not os.path.exists(HETERO_GRAPH_PATH):
    print(f"Error: Graph file not found at {HETERO_GRAPH_PATH}. Please run Step 2 first.")
else:
    print(f"Loading graph from {HETERO_GRAPH_PATH}...")
    # M1 Note: 'cpu' map_location is usually safer primarily, moving to device later
    data = torch.load(HETERO_GRAPH_PATH, map_location="cpu", weights_only=False)
    
    # Optional: Make graph undirected if edges are only one-way but flow is bidirectional
    data = ToUndirected()(data)
    
    print("Graph Loaded Successfully.")
    print(data)

In [ ]:
# Initialize Dummy Features (if missing) just to make Loader work for demo
for node_type in data.node_types:
    if not hasattr(data[node_type], 'x') or data[node_type].x is None:
        data[node_type].x = torch.randn(data[node_type].num_nodes, 16)

In [ ]:
# Define Input Nodes (Seed Nodes)
# Typically we sample 'pekerja' or 'transaksi' nodes for training
# Here we just take the first 1000 'pekerja' nodes as an example
input_nodes = ('pekerja', torch.arange(min(1000, data['pekerja'].num_nodes)))

# Create Loader
loader = NeighborLoader(
    data,
    num_neighbors={key: NUM_NEIGHBORS for key in data.edge_types},
    input_nodes=input_nodes,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    persistent_workers=PERSISTENT_WORKERS,
    pin_memory=PIN_MEMORY
)

print("Loader initialized.")

In [ ]:
# Benchmark Loading Speed
print("Benchmarking...")
start = time.time()
count = 0
for batch in tqdm(loader):
    # Simulate transfer to GPU (MPS)
    batch = batch.to(DEVICE)
    count += 1
end = time.time()

print(f"Processed {count} batches in {end - start:.2f} seconds.")